# scVI Stage 1, two continuations: Leiden/SEACells vs. scProto Stage 2

**What this answers (reviewer F5RB).**
1. *"The cross-batch alignment seems to come mainly from the Stage-1 scPoli-style pretraining ... the training design does not fully support the mechanism claimed in the paper."*
2. *"The most important baseline is missing: first apply a batch-correction method such as scVI, then run an existing graph-based metacell method on the corrected latent space."*
3. Round 3: *"restricting all methods to d=8 ... replacing scVI's standard count likelihood with a Gaussian likelihood may disadvantage methods under nonstandard settings."*

All three are answered by one controlled experiment. Train **plain scVI at its own default configuration** (n_latent=10, ZINB count likelihood, raw counts), then keep everything downstream identical and change exactly one thing:

| arm | Stage 1 | continuation |
|---|---|---|
| A1 | scVI (default) | **Leiden** on its latent |
| A2 | scVI (default) | **SEACells** on its latent |
| B  | scVI (default) | **scProto Stage 2** keeps training that same encoder |

Same pretrained weights (loaded once, in one process), same adaptive-RBF affinity graph, same K, same evaluation code. No scPoli anywhere. Any difference between A and B is the Stage-2 objective and nothing else.

There is also a fourth, free control: **epoch 0** of arm B — prototypes waypoint-initialised on the *frozen* scVI latent, before a single gradient step — printed by the training cell as `[Epoch 0] initial modularity=...`. It is the "no Stage-2 training at all" reference point.

**Code.** `interpretable_ssl/models/scvi_backbone.py` (scVI-backed scProto model), `interpretable_ssl/trainers/scvi_proto.py` (`ScviProtoTrainer` = `SCProtoTrainer` with Stage 1 replaced), `interpretable_ssl/experiments/scvi_stage2.py` (the runner used below). Stage 2 itself — waypoint init, temperature calibration, community loss, nassoc, usage, prototype reconstruction, early stopping on modularity — is inherited unchanged from the paper's own trainer, and evaluation loads the best checkpoint exactly like `find_metacells` does.

**Resuming.** Every stage writes to Drive and reloads if present (scVI weights → `models/{ds}/scvi_stage1/`, Stage-1 model state → `models/{ds}/pretrain_scvi/`, Stage-2 → `umap_checkpoint.pth` in the run dir, baselines → their own `metrics.json`). A dataset that fails is skipped, not fatal. After a disconnect, re-run the cells top to bottom: finished work reloads in seconds and training continues from the last evaluation.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Same install cell as batch_correct_then_cluster_baselines.ipynb / headline_significance_test.ipynb.
# Only needed once per fresh runtime. RESTART THE RUNTIME after this cell before running
# anything below -- numpy/scipy/anndata are C-extension linked, an in-process upgrade
# won't reliably take effect on already-imported modules.
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

In [22]:
# Ground truth for "did the install actually work" -- pip's log is noisy, importing is
# the real test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {getattr(mod, '__version__', '?')})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")
print(f"\n{len(_failed)} failed: {_failed}" if _failed else f"\nAll {len(_checks)} packages import cleanly.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)
  OK   anndata          (import anndata, version 0.13.2)
  OK   scanpy           (import scanpy, version 1.12.3)
  OK   scarches         (import scarches, version 0.6.1)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   faiss-cpu        (import faiss, version 1.14.1)

All 13 packages import cleanly.


In [23]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [24]:
import os, json, glob
import numpy as np
import pandas as pd

from interpretable_ssl.experiments.scvi_stage2 import (
    SCVI_DEF_TAG,
    run_all_datasets,
    run_scvi_stage2_experiment,
    get_scvi_proto_trainer,
    run_scvi_clustering_baselines,
    resolve_run_names,
    collect_task1_table,
)
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir
from interpretable_ssl.evaluation.paper_figures import (
    rare_celltype_purity_table, rare_metric_significance_paired,
)

print("scvi_stage2 imports ready")

scvi_stage2 imports ready


## Config

In [25]:
# All three RNA-seq datasets. To (re-)run just one, set this to e.g. ['lung'] --
# every finished stage reloads from disk, so nothing is recomputed.
DATASETS_TO_RUN = ['pancreas', 'lung', 'pbmc-immune']
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# scVI at its OWN defaults -- that is the whole point of this run (round-3 comment:
# d=8 + Gaussian likelihood was a nonstandard setting for scVI). n_latent=10 and
# gene_likelihood='zinb' are scvi-tools' defaults; scProto's Stage 2 adapts to them
# (the prototype layer is built at scVI's latent dimension), not the other way round.
SCVI_N_LATENT        = 10
SCVI_GENE_LIKELIHOOD = 'zinb'
SCVI_EPOCHS          = 50    # Stage 1 budget
STAGE2_MAX_EPOCHS    = 20    # Stage 2 budget (early stopping on modularity may end sooner)
EVAL_FREQ            = 3
PATIENCE             = 6
BATCH_SIZE           = 1024
UMAP_STEPS_PER_EPOCH = 500

for ds in DATASETS_TO_RUN:
    cfg = DATASETS[ds]
    print(f"{ds:<14} K={cfg['num_prototypes']:<5} label_key={cfg['label_key']:<18} batch_key={cfg.get('batch_key')}")

pancreas       K=220   label_key=celltype           batch_key=tech
lung           K=300   label_key=cell_type          batch_key=batch
pbmc-immune    K=300   label_key=final_annotation   batch_key=study


In [26]:
# Preflight: the adaptive-RBF affinity graph should already be on disk for each dataset.
#
# Why check instead of just letting it build: the graph filename is keyed only on
# (dataset, n_cells, n_comps, k, affinity_type) -- not on what adata.X held when it was
# built. This experiment feeds the model log1p(counts) (scVI's own encoder input space),
# so a MISSING graph would be generated from log1p(counts) PCA and would no longer be
# the same graph every other method in the rebuttal was scored on. Present = every arm
# here shares the canonical graph, which is what makes the comparison clean.
for ds in DATASETS_TO_RUN:
    found = sorted(glob.glob(os.path.join('./graphs', f'affinity_{ds}[0-9]*_ncomp50_kneighbors50_arbf.pkl')))
    if found:
        for g in found:
            print(f"  OK   {ds:<14} {os.path.basename(g)}  ({os.path.getsize(g)/1e6:.1f} MB)")
    else:
        print(f"  MISS {ds:<14} no cached ARBF graph -- it will be built from this run's "
              f"adata (log1p counts PCA). Arms stay internally consistent, but modularity "
              f"would not be directly comparable to the other rebuttal tables.")

  OK   pancreas       affinity_pancreas16382_ncomp50_kneighbors50_arbf.pkl  (14.1 MB)
  OK   lung           affinity_lung32472_ncomp50_kneighbors50_arbf.pkl  (29.9 MB)
  OK   pbmc-immune    affinity_pbmc-immune33506_ncomp50_kneighbors50_arbf.pkl  (31.6 MB)


## Run — Stage 1 + both arms, per dataset

For each dataset: train/reload scVI on raw counts, cluster its latent with Leiden and SEACells, then run scProto Stage 2 on that same encoder and evaluate. A dataset that errors is reported and skipped so the others still finish.

Watch for `[Epoch 0] initial modularity=` in each dataset's Stage-2 output — that is the frozen-encoder control.

In [27]:
RESULTS = run_all_datasets(
    DATASETS_TO_RUN,
    scvi_epochs=SCVI_EPOCHS,
    scvi_n_latent=SCVI_N_LATENT,
    scvi_gene_likelihood=SCVI_GENE_LIKELIHOOD,
    stage2_max_epochs=STAGE2_MAX_EPOCHS,
    eval_freq=EVAL_FREQ,
    patience=PATIENCE,
    batch_size=BATCH_SIZE,
    umap_steps_per_epoch=UMAP_STEPS_PER_EPOCH,
    skip_if_exists=True,
)


=== pancreas ===
dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
[pancreas] counts check: max=1453667, integer fraction=0.950
[pancreas] NOTE: 5.0% of non-zero entries are not whole numbers. Values are still count-scale (max=1453667), and this is the same matrix the existing scVI baselines used, so the run proceeds.
    celseq          max=      1597.0  integer fraction=0.888
    celseq2         max=      1597.0  integer fraction=0.840
    fluidigmc1      max=   1453667.0  integer fraction=0.851
    smartseq2       max=   1341375.0  integer fraction=1.000
    inDrop1         max=      4318.0  integer fraction=1.000
    inDrop2         max=      3476.0  integer fraction=1.000
    inDrop3         max=      3071.0  integer fraction=1.000
    inDrop4         max=      4234.0  integer fraction=1.000
    smarter         max=    443598.2  integer fraction=0.839
[pancreas] adata.X set to log1p(counts) -- scVI's own encoder input space; raw cou

  0%|          | 0/16 [00:00<?, ?it/s]

[pancreas] scVI latent: (16382, 10)

=== [pancreas] arm A: scVI latent -> Leiden / SEACells ===
Computing kNN graph using scanpy NN ...
Computing radius for adaptive bandwidth kernel...


  0%|          | 0/16382 [00:00<?, ?it/s]

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


...

Making graph symmetric...
Parameter graph_construction = union being used to build KNN graph...
Computing RBF kernel...


  0%|          | 0/33506 [00:00<?, ?it/s]

Building similarity LIL matrix...


  0%|          | 0/33506 [00:00<?, ?it/s]

Constructing CSR matrix...
[pbmc-immune] embedding-level rare affinity purity saved to /content/drive/MyDrive/models/pbmc-immune/scvi_stage2_affinity_purity_pbmc-immune.json

[pbmc-immune] done. Stage-2 run dir: /content/drive/MyDrive/models//pbmc-immune/scviproto_LD10_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31

finished: ['pancreas', 'lung', 'pbmc-immune']  failed: []


In [28]:
# OPTIONAL control (off by default -- one extra Leiden run per dataset).
#
# Arm B trains on the unit sphere (l2norm=1, dot-product assignment) because that is
# scProto's own Stage-2 recipe from the paper, while arm A clusters scVI's latent
# as-is, the way anyone using scVI would. Flip this on to also cluster the L2-normalised
# scVI latent, which rules out "the difference is just the normalisation".
# Tag 'X_scvil2' deliberately does not start with 'X_scvidef' -- a prefix-colliding tag
# would make the 'leiden_X_scvidef' keyword ambiguous in the tables below.
RUN_L2NORM_CONTROL = False

if RUN_L2NORM_CONTROL:
    for ds in DATASETS_TO_RUN:
        res = RESULTS.get(ds)
        if res is None:
            continue
        t_ds = res['trainer']
        run_scvi_clustering_baselines(
            t_ds, t_ds.get_latent(l2norm=True), tag='X_scvil2',
            run_seacells=False, run_leiden=True, skip_if_exists=True,
        )

## Results

One keyword set covers all datasets: `extract_model_key` strips the dataset-specific tokens (`_ds-*`, `_NP*`, `_v*`) so each dataset's Stage-2 run maps to the same row, and the Leiden keyword omits `_K{n}` because K differs per dataset.

The canonical scProto runs (scPoli Stage 1, d=8) are included for context only — a different Stage 1 and a different latent dimension, so the controlled comparison is the three scVI rows among themselves.

In [29]:
_first = next((r for r in RESULTS.values() if r is not None), None)
assert _first is not None, "no dataset finished -- nothing to summarise"

MODEL_KEYWORDS = resolve_run_names(_first['run_dir'], include_canonical=True)
MODEL_KEYWORDS

{'seacell_X_scvidef': 'SEACells (scVI)',
 'leiden_X_scvidef': 'Leiden (scVI)',
 'scviproto_LD10_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto Stage 2 (scVI)',
 'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp': 'scProto (scPoli Stage 1)'}

In [30]:
df_rare = rare_celltype_purity_table(DATASETS_TO_RUN, model_keywords=MODEL_KEYWORDS, verbose=False)

_dupe = df_rare.index.duplicated(keep='first')
if _dupe.any():
    print(f"WARNING: dropped {_dupe.sum()} duplicate row(s): {df_rare.index[_dupe].tolist()}")
    df_rare = df_rare[~_dupe]

show_table(
    df_rare,
    metrics=[
        'batch_rare_coverage_mean', 'batch_rare_recall_macro_mean',
        'batch_rare_precision_macro_mean', 'batch_rare_homogeneity_mean',
        'batch_rare_cross_batch_homog_mean', 'batch_rare_f1_macro_mean',
    ],
    dataset_display_names=dataset_display_names,
)

  [SEACells (scVI)|pancreas] resolving run dir ...
  [Leiden (scVI)|pancreas] resolving run dir ...
  [scProto Stage 2 (scVI)|pancreas] resolving run dir ...
  [scProto (scPoli Stage 1)|pancreas] resolving run dir ...
  [SEACells (scVI)|lung] resolving run dir ...
  [Leiden (scVI)|lung] resolving run dir ...
  [scProto Stage 2 (scVI)|lung] resolving run dir ...
  [scProto (scPoli Stage 1)|lung] resolving run dir ...
  [SEACells (scVI)|lung] run dir resolved (0.1s)
  [Leiden (scVI)|lung] run dir resolved (0.1s)
  [scProto Stage 2 (scVI)|lung] run dir resolved (0.1s)
  [SEACells (scVI)|lung] reading umap_cells.csv ...
  [Leiden (scVI)|lung] reading umap_cells.csv ...
  [scProto Stage 2 (scVI)|lung] reading umap_cells.csv ...
  [SEACells (scVI)|lung] umap_cells.csv loaded (32472 rows, 0.0s)
  [SEACells (scVI)|lung] reading umap_protos.csv ...
  [Leiden (scVI)|lung] umap_cells.csv loaded (32472 rows, 0.1s)
  [Leiden (scVI)|lung] reading umap_protos.csv ...  [scProto Stage 2 (scVI)|lung] um

In [31]:
# Mean +/- std across batches, per dataset, for the two headline rare metrics.
for metric in ['batch_rare_f1_macro', 'batch_rare_homogeneity']:
    rows = []
    for (ds, run) in df_rare.index:
        vals = np.asarray(df_rare.loc[(ds, run), f'_{metric}_per_batch'], dtype=float)
        rows.append({'dataset': dataset_display_names.get(ds, ds), 'method': run,
                     'n_batches': len(vals),
                     'mean': round(float(vals.mean()), 3),
                     'std': round(float(vals.std()), 3),
                     'median': round(float(np.median(vals)), 3)})
    print(f"=== {metric} (mean +/- std across batches) ===")
    display(pd.DataFrame(rows).pivot(index='method', columns='dataset',
                                     values=['mean', 'std', 'n_batches']))

=== batch_rare_f1_macro (mean +/- std across batches) ===


mean                    std                  \
dataset                  Immune   Lung Pancreas Immune   Lung Pancreas   
method                                                                   
Leiden (scVI)             0.855  0.689    0.258  0.108  0.167    0.276   
SEACells (scVI)           0.892  0.682    0.639  0.049  0.162    0.196   
scProto (scPoli Stage 1)  0.854  0.599    0.535  0.138  0.176    0.208   
scProto Stage 2 (scVI)    0.897  0.557    0.614  0.030  0.255    0.218   

                         n_batches                 
dataset                     Immune  Lung Pancreas  
method                                             
Leiden (scVI)                  5.0  15.0      8.0  
SEACells (scVI)                5.0  15.0      8.0  
scProto (scPoli Stage 1)       5.0  15.0      8.0  
scProto Stage 2 (scVI)         5.0  15.0      8.0

=== batch_rare_homogeneity (mean +/- std across batches) ===


mean                    std                  \
dataset                  Immune   Lung Pancreas Immune   Lung Pancreas   
method                                                                   
Leiden (scVI)             0.729  0.635    0.317  0.158  0.182    0.242   
SEACells (scVI)           0.819  0.658    0.588  0.082  0.180    0.180   
scProto (scPoli Stage 1)  0.827  0.578    0.565  0.102  0.221    0.155   
scProto Stage 2 (scVI)    0.859  0.560    0.601  0.064  0.257    0.173   

                         n_batches                 
dataset                     Immune  Lung Pancreas  
method                                             
Leiden (scVI)                  5.0  15.0      8.0  
SEACells (scVI)                5.0  15.0      8.0  
scProto (scPoli Stage 1)       5.0  15.0      8.0  
scProto Stage 2 (scVI)         5.0  15.0      8.0

In [32]:
# Paired one-sided Wilcoxon (scProto Stage 2 > baseline) on the same batches,
# Bonferroni-corrected per dataset.
df_sig = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto Stage 2 (scVI)',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
        '_batch_rare_cross_batch_homog_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)

for metric_name in df_sig['metric'].unique():
    print(f"=== {metric_name}: scProto Stage 2 (scVI) vs. each same-K arm, PAIRED "
          f"one-sided Wilcoxon, Bonferroni-corrected ===")
    sub = df_sig[df_sig['metric'] == metric_name].copy()
    sub['cell'] = sub.apply(
        lambda r: f"{r['mean']:.3f}+/-{r['std']:.3f} (K={r['k']}, n={r['n']}) [ref]"
        if r['method'] == 'scProto Stage 2 (scVI)'
        else f"{r['mean']:.3f}+/-{r['std']:.3f} (K={r['k']}, n={r['n']}, "
             f"wins={r.get('n_wins', '?')}/{r['n']})  {r.get('sig', '?')}  "
             f"p_adj={r.get('p_adj', float('nan')):.3g}",
        axis=1,
    )
    display(sub.pivot(index='method', columns='dataset', values='cell'))

df_sig

=== batch_rare_f1_macro: scProto Stage 2 (scVI) vs. each same-K arm, PAIRED one-sided Wilcoxon, Bonferroni-corrected ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (scVI),"0.855+/-0.108 (K=300, n=5, wins=4.0/5) ns p_...","0.689+/-0.167 (K=300, n=15, wins=4.0/15) ns ...","0.258+/-0.276 (K=220, n=8, wins=8.0/8) * p_a..."
SEACells (scVI),"0.892+/-0.049 (K=300, n=5, wins=3.0/5) ns p_...","0.682+/-0.162 (K=300, n=15, wins=1.0/15) ns ...","0.639+/-0.196 (K=220, n=8, wins=3.0/8) ns p_..."
scProto (scPoli Stage 1),"0.854+/-0.138 (K=294, n=5, wins=2.0/5) ns p_...","0.599+/-0.176 (K=298, n=15, wins=8.0/15) ns ...","0.535+/-0.208 (K=219, n=8, wins=6.0/8) ns p_..."
scProto Stage 2 (scVI),"0.897+/-0.030 (K=293, n=5) [ref]","0.557+/-0.255 (K=297, n=15) [ref]","0.614+/-0.218 (K=220, n=8) [ref]"


=== batch_rare_homogeneity: scProto Stage 2 (scVI) vs. each same-K arm, PAIRED one-sided Wilcoxon, Bonferroni-corrected ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (scVI),"0.729+/-0.158 (K=300, n=5, wins=5.0/5) ns p_...","0.635+/-0.182 (K=300, n=15, wins=4.0/15) ns ...","0.317+/-0.242 (K=220, n=8, wins=8.0/8) * p_a..."
SEACells (scVI),"0.819+/-0.082 (K=300, n=5, wins=4.0/5) ns p_...","0.658+/-0.180 (K=300, n=15, wins=2.0/15) ns ...","0.588+/-0.180 (K=220, n=8, wins=3.0/8) ns p_..."
scProto (scPoli Stage 1),"0.827+/-0.102 (K=294, n=5, wins=4.0/5) ns p_...","0.578+/-0.221 (K=298, n=15, wins=7.0/15) ns ...","0.565+/-0.155 (K=219, n=8, wins=5.0/8) ns p_..."
scProto Stage 2 (scVI),"0.859+/-0.064 (K=293, n=5) [ref]","0.560+/-0.257 (K=297, n=15) [ref]","0.601+/-0.173 (K=220, n=8) [ref]"


=== batch_rare_cross_batch_homog: scProto Stage 2 (scVI) vs. each same-K arm, PAIRED one-sided Wilcoxon, Bonferroni-corrected ===


dataset,Immune,Lung,Pancreas
method,,,
Leiden (scVI),"0.407+/-0.208 (K=300, n=5, wins=2.0/5) ns p_...","0.597+/-0.159 (K=300, n=15, wins=3.0/15) ns ...","0.237+/-0.146 (K=220, n=8, wins=4.0/8) ns p_..."
SEACells (scVI),"0.335+/-0.194 (K=300, n=5, wins=2.0/5) ns p_...","0.598+/-0.154 (K=300, n=15, wins=3.0/15) ns ...","0.411+/-0.123 (K=220, n=8, wins=0.0/8) ns p_..."
scProto (scPoli Stage 1),"0.349+/-0.198 (K=294, n=5, wins=3.0/5) ns p_...","0.508+/-0.197 (K=298, n=15, wins=6.0/15) ns ...","0.323+/-0.179 (K=219, n=8, wins=2.0/8) ns p_..."
scProto Stage 2 (scVI),"0.303+/-0.195 (K=293, n=5) [ref]","0.466+/-0.242 (K=297, n=15) [ref]","0.205+/-0.133 (K=220, n=8) [ref]"


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,SEACells (scVI),220,8,0.682488,0.638570,0.196148,3.0,0.808594,1.000000,ns
1,Pancreas,batch_rare_f1_macro,Leiden (scVI),220,8,0.185185,0.257980,0.276038,8.0,0.003906,0.011719,*
2,Pancreas,batch_rare_f1_macro,scProto Stage 2 (scVI),220,8,0.596716,0.613513,0.217825,NaN,NaN,NaN,NaN
3,Pancreas,batch_rare_f1_macro,scProto (scPoli Stage 1),219,8,0.437680,0.535403,0.207961,6.0,0.156250,0.468750,ns
4,Pancreas,batch_rare_homogeneity,SEACells (scVI),220,8,0.600245,0.588033,0.179627,3.0,0.628906,1.000000,ns
5,Pancreas,batch_rare_homogeneity,Leiden (scVI),220,8,0.248668,0.316738,0.242439,8.0,0.003906,0.011719,*
6,Pancreas,batch_rare_homogeneity,scProto Stage 2 (scVI),220,8,0.604024,0.600766,0.173070,NaN,NaN,NaN,NaN
7,Pancreas,batch_rare_homogeneity,scProto (scPoli Stage 1),219,8,0.522129,0.564854,0.155263,5.0,0.230469,0.691406,ns
8,Pancreas,batch_rare_cross_batch_homog,SEACells (scVI),220,8,0.404908,0.411440,0.122933,0.0,1.000000,1.000000,ns
9,Pancreas,batch_rare_cross_batch_homog,Leiden (scVI),220,8,0.209340,0.236849,0.146107,4.0,0.527344,1.000000,ns


## Table 1 metrics for every arm and dataset

Purity / batch entropy / modularity / coverage, read from each run's own `metrics.json`. All arms are rescored against the same canonical graph.

In [33]:
df_task1 = collect_task1_table(DATASETS_TO_RUN, MODEL_KEYWORDS, dataset_display_names)
display(df_task1.pivot(index='method', columns='dataset',
                       values=['modularity', 'purity', 'batch_entropy', 'coverage']))
df_task1

modularity                        purity            \
dataset                      Immune      Lung  Pancreas    Immune      Lung   
method                                                                        
Leiden (scVI)              0.364379  0.522427  0.379159  0.867663  0.850151   
SEACells (scVI)            0.240445  0.326896  0.295673  0.894180  0.860817   
scProto (scPoli Stage 1)   0.628591  0.664444  0.601234  0.901103  0.858799   
scProto Stage 2 (scVI)     0.618066  0.650069  0.615681  0.892998  0.872173   

                                   batch_entropy                     coverage  \
dataset                   Pancreas        Immune      Lung  Pancreas   Immune   
method                                                                          
Leiden (scVI)             0.964776      0.806087  1.185238  0.769753   1.0000   
SEACells (scVI)           0.954860      0.526690  1.110295  0.810051   1.0000   
scProto (scPoli Stage 1)  0.912529      0.229215  0.537913  0.399301   0.9375   
scProto Stage 2 (scVI)    0.913605      0.223084  0.558935  0.300580   0.9375   

                                              
dataset                       Lung  Pancreas  
method                                        
Leiden (scVI)             1.000000  0.785714  
SEACells (scVI)           1.000000  1.000000  
scProto (scPoli Stage 1)  0.882353  0.928571  
scProto Stage 2 (scVI)    1.000000  1.000000

,dataset,method,purity,batch_entropy,modularity,modularity_std,coverage,K
0,Pancreas,SEACells (scVI),0.954860,0.810051,0.295673,0.077863,1.000000,220.0
1,Pancreas,Leiden (scVI),0.964776,0.769753,0.379159,0.121646,0.785714,220.0
2,Pancreas,scProto Stage 2 (scVI),0.913605,0.300580,0.615681,0.081900,1.000000,220.0
3,Pancreas,scProto (scPoli Stage 1),0.912529,0.399301,0.601234,0.088443,0.928571,NaN
4,Lung,SEACells (scVI),0.860817,1.110295,0.326896,0.052737,1.000000,300.0
5,Lung,Leiden (scVI),0.850151,1.185238,0.522427,0.077166,1.000000,300.0
6,Lung,scProto Stage 2 (scVI),0.872173,0.558935,0.650069,0.039418,1.000000,300.0
7,Lung,scProto (scPoli Stage 1),0.858799,0.537913,0.664444,0.023622,0.882353,NaN
8,Immune,SEACells (scVI),0.894180,0.526690,0.240445,0.129763,1.000000,300.0
9,Immune,Leiden (scVI),0.867663,0.806087,0.364379,0.182858,1.000000,300.0


## Did the latent itself change?

Clustering-free diagnostic: for every locally-rare-type cell, the fraction of its affinity mass that stays within its own cell type, per batch — computed on the scVI latent before and after Stage 2.

In [34]:
for ds, res in RESULTS.items():
    if res is None:
        continue
    print(f"=== {dataset_display_names.get(ds, ds)} ===")
    display(res['embedding_purity'])

=== Pancreas ===


,embedding,dim,n_batches,mean,std,n_wins,p_vs_ref
0,X_scvidef,10,8,0.5148,0.1696,2.0,0.9453
1,X_scviproto,10,8,0.4727,0.1473,NaN,NaN


=== Lung ===


,embedding,dim,n_batches,mean,std,n_wins,p_vs_ref
0,X_scvidef,10,15,0.6554,0.1467,4.0,0.9681
1,X_scviproto,10,15,0.5944,0.2134,NaN,NaN


=== Immune ===


,embedding,dim,n_batches,mean,std,n_wins,p_vs_ref
0,X_scvidef,10,5,0.8684,0.0472,2.0,0.7812
1,X_scviproto,10,5,0.8634,0.0473,NaN,NaN


## Save a compact summary

In [35]:
summary = {
    'datasets': DATASETS_TO_RUN,
    'scvi': {'n_latent': SCVI_N_LATENT, 'gene_likelihood': SCVI_GENE_LIKELIHOOD,
             'epochs': SCVI_EPOCHS},
    'stage2': {'max_epochs': STAGE2_MAX_EPOCHS, 'eval_freq': EVAL_FREQ, 'patience': PATIENCE},
    'arms': MODEL_KEYWORDS,
    'per_dataset': {
        ds: {
            'K': int(DATASETS[ds]['num_prototypes']),
            'run_dir': res['run_dir'],
            'stage2_metrics': res['stage2_metrics'],
            'embedding_rare_affinity_purity': res['embedding_purity'].to_dict(orient='records'),
        }
        for ds, res in RESULTS.items() if res is not None
    },
    'task1': df_task1.to_dict(orient='records'),
    'rare_significance': df_sig.to_dict(orient='records'),
}

out_dir = '/content/drive/MyDrive/codes/interpretable-prototype/neurips_manuscript/rebuttle/experiment-results'
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'scvi_stage2_summary.json')
with open(out_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print("saved", out_path)

saved /content/drive/MyDrive/codes/interpretable-prototype/neurips_manuscript/rebuttle/experiment-results/scvi_stage2_summary.json
